# gai4sa — Semantic ADR Clustering

Clusters Architectural Decision Records using sentence-transformers embeddings.
Run this in Google Colab (Runtime → Run all) or any Jupyter environment.

**Setup:** Upload `adr_dataset.jsonl` to Colab before running, or mount Google Drive.

In [ ]:
# Install dependencies (run once per session)
!pip install sentence-transformers umap-learn hdbscan 2>&1 | tail -3

In [ ]:
import json
import re
from pathlib import Path
from collections import Counter

import hdbscan
import umap
from sentence_transformers import SentenceTransformer

print("All imports OK")

In [ ]:
# ── Load data ──────────────────────────────────────────────────────
# Try multiple locations: local upload, Drive, or repo path
candidates = [
    "adr_dataset.jsonl",
    "/content/adr_dataset.jsonl",
    "/content/drive/MyDrive/adr_dataset.jsonl",
    "drive/MyDrive/adr_dataset.jsonl",
]

jsonl_path = None
for p in candidates:
    if Path(p).exists():
        jsonl_path = p
        break

if not jsonl_path:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        for p in ["/content/drive/MyDrive/adr_dataset.jsonl"]:
            if Path(p).exists():
                jsonl_path = p
                break
    except ImportError:
        pass

if not jsonl_path:
    raise FileNotFoundError(
        "Upload adr_dataset.jsonl to Colab or place it in Google Drive/MyDrive"
    )

with open(jsonl_path) as f:
    records = [json.loads(l) for l in f]
print(f"Loaded {len(records)} records from {jsonl_path}")

In [ ]:
# ── Schema validation ──────────────────────────────────────────────
# Verify the expected keys exist before embedding
required_key = "raw_text"
available_keys = list(records[0].keys())
print(f"Record keys: {available_keys}")

if required_key not in available_keys:
    # Show first record to help debug
    print("First record contents:")
    print(json.dumps(records[0], indent=2)[:500])
    raise KeyError(
        f"Expected '{required_key}' in JSONL records. "
        f"Found keys: {available_keys}. "
        "Update the 'required_key' variable above to match your data."
    )
print("Schema OK")

In [ ]:
# ── Embed with sentence-transformers ───────────────────────────────
model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Model loaded: all-MiniLM-L6-v2 (dim={model.get_sentence_embedding_dimension()})")

# Clean text slightly before embedding
def clean_text(text: str) -> str:
    text = re.sub(r'[=\-~]{3,}', ' ', text)
    text = re.sub(r'`{2,}', ' ', text)
    return text

texts = [clean_text(r[required_key]) for r in records]
embeddings = model.encode(texts, show_progress_bar=True)
print(f"Embedding shape: {embeddings.shape}")

In [ ]:
# ── Reduce dimensionality ──────────────────────────────────────────
reducer = umap.UMAP(n_components=5, random_state=42, n_neighbors=15, min_dist=0.0)
embeddings_umap = reducer.fit_transform(embeddings)
print(f"Reduced shape: {embeddings_umap.shape}")

In [ ]:
# ── Cluster with HDBSCAN ───────────────────────────────────────────
# Tune these parameters:
#   min_cluster_size — smaller = more clusters, larger = fewer.
#                      For datasets under ~50 records, try 2.
#   min_samples     — higher = more conservative (more noise).
#                      Usually keep at 1-3.
MIN_CLUSTER_SIZE = 3
MIN_SAMPLES = 2

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=MIN_SAMPLES,
    metric="euclidean",
    prediction_data=True,
)
labels = clusterer.fit_predict(embeddings_umap)
probs = clusterer.probabilities_

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise = list(labels).count(-1)
print(f"min_cluster_size={MIN_CLUSTER_SIZE}, min_samples={MIN_SAMPLES}")
print(f"Clusters: {n_clusters}, Noise: {noise}/{len(records)}")

In [ ]:
# ── Cluster summary ────────────────────────────────────────────────
# (source prefix assumes filenames like "edx_0001-foo.rst" or
#  "backstage_adr007-bar.md" — adjust split logic if your naming differs)
for label in sorted(set(labels)):
    members = [i for i, l in enumerate(labels) if l == label]
    srcs = Counter(records[i]["source_file"].split("_")[0] for i in members)
    print(f"\n  Cluster {label} ({len(members)}): {dict(srcs)}")
    for i in members[:5]:
        ctx = (records[i].get("context") or "")[:100]
        print(f"    {records[i]['source_file']}: {ctx}...")
    if len(members) > 5:
        print(f"    ... +{len(members)-5} more")

In [ ]:
# ── Save results ───────────────────────────────────────────────────
output = []
for i, r in enumerate(records):
    output.append({
        "source_file": r["source_file"],
        "cluster": int(labels[i]),
        "cluster_prob": float(probs[i]) if probs[i] is not None else 0.0,
        "context_preview": (r.get("context") or "")[:120],
    })

out_path = Path(jsonl_path).parent / "adr_clusters_semantic.json"
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)
print(f"Saved to {out_path}")

# Download trigger (Colab-only — safe fallback for local runs)
try:
    from google.colab import files
    files.download(str(out_path))
    print("Done. Download should start automatically.")
except ImportError:
    print("Not in Colab — file saved locally. Download it manually.")